<a href="https://colab.research.google.com/github/RomanLuja/ML/blob/main/project_fixed_errors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Типові помилки і як їх знайти

## 1. Знайди помилку

У кожному завданні — код, який **запускається без помилок**, але дає неправильний або оманливий
результат. Знайдіть проблему, поясніть її в коментарі й виправте.

In [ ]:
# Підготовка: бібліотеки та функція завантаження даних (працює в Colab і локально)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
DATA_URL = "https://raw.githubusercontent.com/KsiuTretyakova/ml_for_analys_dta_2026/refs/heads/main/data/"


def load_csv(name):
    """Читає CSV з поточної теки, з теки data/, з DATA_URL або просить завантажити файл у Colab."""
    for path in (name, os.path.join("data", name)):
        if os.path.exists(path):
            return pd.read_csv(path)
    if DATA_URL:
        return pd.read_csv(DATA_URL.rstrip("/") + "/" + name)
    from google.colab import files
    print(f"Натисніть «Choose files» і виберіть файл {name}")
    files.upload()
    return pd.read_csv(name)

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, mean_absolute_error

loans = load_csv("bank_loans.csv")
num = ["age", "monthly_income_uah", "years_employed", "loan_amount_uah", "term_months", "interest_rate",
       "credit_history_years", "prior_defaults", "open_loans", "has_collateral"]
X_tr, X_te, y_tr, y_te = train_test_split(loans[num], loans["default"], test_size=0.25,
                                          random_state=42, stratify=loans["default"])

### Помилка 1. «Модель точна на 84 %!»

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
print(f"Accuracy: {accuracy_score(y_te, tree.predict(X_te)):.1%} - чудова модель!")

Accuracy: 84.3% - чудова модель!


> Ваша відповідь

### Помилка 2. Найкраща глибина дерева

In [ ]:
best_depth, best_auc = None, 0
for d in range(1, 15):
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])
    if auc > best_auc:
        best_depth, best_auc = d, auc
print(f"Найкраща глибина {best_depth}, test AUC = {best_auc:.3f} - це наша оцінка якості")

Найкраща глибина 5, test AUC = 0.820 - це наша оцінка якості


> Ваша відповідь

### Помилка 3. Масштабування

In [ ]:
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = StandardScaler().fit_transform(X_te)
lr = LogisticRegression(max_iter=1000).fit(X_tr_s, y_tr)
print(f"AUC: {roc_auc_score(y_te, lr.predict_proba(X_te_s)[:, 1]):.3f}")

AUC: 0.860


> Ваша відповідь

### Помилка 4. Чудова ознака для повернень

In [ ]:
orders = load_csv("online_orders.csv")
orders["customer_return_rate"] = orders.groupby("customer_id")["returned"].transform("mean")
o_tr, o_te = train_test_split(orders, test_size=0.25, random_state=42, stratify=orders["returned"])
feats = ["discount_pct", "delivery_days", "customer_prev_orders", "customer_return_rate"]
m4 = LogisticRegression(max_iter=1000).fit(o_tr[feats], o_tr["returned"])
print(f"AUC: {roc_auc_score(o_te['returned'], m4.predict_proba(o_te[feats])[:, 1]):.3f} - ознака customer_return_rate дуже допомагає!")

AUC: 0.915 - ознака customer_return_rate дуже допомагає!


> Ваша відповідь

### Помилка 5. Прогноз попиту

In [ ]:
bikes = load_csv("bike_rentals_daily.csv").sort_values("date").reset_index(drop=True)
bikes["day_index"] = range(len(bikes))
bf = ["temperature_c", "humidity_pct", "wind_kmh", "is_weekend", "day_index"]
b_tr, b_te = train_test_split(bikes, test_size=0.25, random_state=42)
m5 = LinearRegression().fit(b_tr[bf], b_tr["rentals"])
print(f"MAE: {mean_absolute_error(b_te['rentals'], m5.predict(b_te[bf])):.0f} прокатів - так і буде в майбутньому")

MAE: 170 прокатів - так і буде в майбутньому


> Ваша відповідь

### Помилка 6. Висновок для керівника

*«Модель показала, що клієнти з заставою рідше не повертають кредит. Тому пропонуємо вимагати
заставу від усіх клієнтів — це зменшить дефолти на 30 %. Модель точна на 86 %.»*

Знайдіть щонайменше три проблеми в цьому висновку.

> Ваша відповідь

## 2. Самоперевірка проєкту

Пройдіться по своєму ноутбуку й позначте кожен пункт.

| # | Перевірка | Як перевірити |
|---|---|---|
| 1 | Ноутбук виконується з нуля | `Середовище виконання → Перезапустити й виконати все` |
| 2 | Немає колонок, відомих лише після події | для кожної колонки: «чи знатиму це в момент прогнозу?» |
| 3 | Результат не «занадто добрий» | AUC > 0.95 або R² > 0.95 — шукайте витік |
| 4 | Уся обробка в `Pipeline` | немає `fit` / `fit_transform` на всіх даних до розділення |
| 5 | Test використано один раз | вибір моделі й параметрів — лише крос-валідацією |
| 6 | Є орієнтир | `Dummy...` або просте правило в журналі експериментів |
| 7 | Схема валідації відповідає даним | час → розділення за часом |
| 8 | `random_state=42` усюди, де є випадковість | повторний запуск дає ті самі числа |
| 9 | Кожне число у висновку взяте з виводу коду | звірте текст з комірками |
| 10 | Висновок у грошах або діях, з обмеженнями | 6–10 речень без жаргону |

## 3. Захист проєкту

1. Задача і замовник: яке рішення допомагає ухвалити модель.
2. Дані: що знайшли в аудиті, що виправили (1–2 найцікавіші проблеми).
3. Моделі: журнал експериментів, обрана модель, якість на test порівняно з орієнтиром.
4. Чому модель так вирішує: 3–4 головні фактори, один приклад пояснення.
5. Бізнес-ефект: як використовувати прогноз, гроші/дії, інтервал.
6. Обмеження і наступні кроки.

Типові запитання:
- *«Як ви перевіряли витоки?»*
- *«Чому саме ця метрика?»*
- *«Що буде, якщо припущення про вартість зміниться?»*
- *«Як часто перенавчати модель?»*